# Binding Spec: Domain Configuration

Every domain in SCPN Phase Orchestrator is described by a **binding spec** —
a YAML file that declares oscillators, coupling, boundaries, actuators,
and the objective partition (good vs bad layers).

This notebook demonstrates:

1. Loading a binding spec from a domainpack
2. Inspecting oscillator families, coupling, and boundaries
3. Running a simulation from the spec
4. Validating a custom binding spec

In [ ]:
from pathlib import Path

import numpy as np

from scpn_phase_orchestrator.binding.loader import load_binding_spec
from scpn_phase_orchestrator.binding.validator import validate_binding_spec
from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

## 1. Load a domainpack binding spec

In [ ]:
spec_path = Path("../domainpacks/minimal_domain/binding_spec.yaml")
spec = load_binding_spec(spec_path)

print(f"Domain: {spec.name} v{spec.version}")
print(f"Safety tier: {spec.safety_tier}")
print(f"Sample period: {spec.sample_period_s}s, Control period: {spec.control_period_s}s")
print(f"Layers: {len(spec.layers)}")
for layer in spec.layers:
    print(f"  {layer.name} (index={layer.index}, oscillators={layer.oscillator_ids})")

## 2. Inspect coupling and oscillator families

In [ ]:
print(f"Coupling: base_strength={spec.coupling.base_strength}, decay_alpha={spec.coupling.decay_alpha}")
print(f"Templates: {spec.coupling.templates or 'none'}")
print()
print("Oscillator families:")
for name, fam in spec.oscillator_families.items():
    print(f"  {name}: channel={fam.channel}, extractor={fam.extractor_type}")
print()
print("Objectives:")
print(f"  Good layers: {spec.objectives.good_layers}")
print(f"  Bad layers:  {spec.objectives.bad_layers}")
print()
print(f"Actuators: {len(spec.actuators)}")
for act in spec.actuators:
    print(f"  {act.name}: knob={act.knob}, scope={act.scope}, limits={act.limits}")

## 3. Build coupling and run simulation from spec

In [ ]:
n_osc = sum(len(layer.oscillator_ids) for layer in spec.layers)
print(f"Total oscillators: {n_osc}")

coupling = CouplingBuilder().build(
    n_osc,
    base_strength=spec.coupling.base_strength,
    decay_alpha=spec.coupling.decay_alpha,
)
print(f"Knm shape: {coupling.knm.shape}")
print(f"Knm (rounded):\n{np.round(coupling.knm, 3)}")

engine = UPDEEngine(n_osc, dt=spec.sample_period_s)
rng = np.random.default_rng(42)
phases = rng.uniform(0, 2 * np.pi, n_osc)
omegas = rng.uniform(0.8, 1.2, n_osc)
alpha = coupling.alpha.copy()

history_r = []
for _ in range(200):
    phases = engine.step(phases, omegas, coupling.knm, 0.0, 0.0, alpha)
    R, _ = compute_order_parameter(phases)
    history_r.append(R)

print(f"\nR after 200 steps: {history_r[-1]:.4f}")
print(f"R range: [{min(history_r):.4f}, {max(history_r):.4f}]")

## 4. Validate a binding spec

The validator checks structural constraints: matching oscillator counts,
valid layer indices, non-negative coupling, and required fields.

In [ ]:
errors = validate_binding_spec(spec)
if errors:
    for e in errors:
        print(f"  ERROR: {e}")
else:
    print("Binding spec is valid.")

print("\nLoading all domainpack specs:")
domainpacks_dir = Path("../domainpacks")
for dp in sorted(domainpacks_dir.iterdir()):
    bs = dp / "binding_spec.yaml"
    if bs.exists():
        s = load_binding_spec(bs)
        errs = validate_binding_spec(s)
        status = "VALID" if not errs else f"{len(errs)} errors"
        print(f"  {s.name}: {status}")

## Summary

- `load_binding_spec(path)` parses YAML/JSON into a typed `BindingSpec`
- `validate_binding_spec(spec)` checks structural invariants
- Each domainpack provides a `binding_spec.yaml` defining oscillator layout, coupling, and boundaries
- The spec drives `CouplingBuilder`, `UPDEEngine`, and `BoundaryObserver` configuration

See `docs/specs/binding_spec.schema.json` for the full JSON Schema, and
`docs/tutorials/01_new_domain_checklist.md` for a step-by-step guide to
creating your own domainpack.